# 20260620 ICD-10 → ICD-11 MMS 实体链接（Entity Linking）复盘

## 0：QA

1. **当前代码解决的任务到底是什么？输入是什么，输出是什么，在做什么任务。**

> 实体链接任务。输入是ICD-10 ICD-11，输出是target数据集的top50


2. **对于实体链接来说，Query是查询项，被检索是Target。哪个是ICD-10， ICD-11？**

> `prepare.py` query是ICD-10，Target是 ICD-11.

3. **训练集、验证集、测试集的 8:1:1，切分的究竟是术语、映射pair，还是某一侧的实体？**

> 以 query ICD-10 的有 match的数量划分的。

4. **Query 和 Target 所在的哪个Knowledge Base 构造了向量数据库？**

> 针对query和target这两个knowledge base, 分别构建两个向量数据库。 

5. **哪些 ICD-11 term 被放进向量数据库：全部 ICD-11，还是仅训练集中的 ICD-11？**

> prepared_dir/source_terms.jsonl 和 target_terms.jsonl。其实是相当于是全量的term的信息都被放在了向量数据库中。

6. **测试时的候选空间是什么：全部 ICD-11、测试集 ICD-11，还是预先筛选的一部分？**

> 全部 ICD-11 （target 数据集）
TODO: 我记得之前有说，BERT path 这个模型，它在测试的时候的候选空间和我的不一样。然后我当时还说，我不确定我的这个研究有没有测试机泄露这个问题好像还没解决

**B. 当前模型结构**


14. 我之前认为的“检索 -> Rerank -> 对比学习”这个粗画像，到底是不是错的？
> 不是这样的，他只是用了对比学习的损失函数，然后去微调了bce embedding 然后用bc embedding对它进行了一个retrieval和rerank

15. Rerank 之后的 Top50，具体是如何送入对比学习中继续更新模型的？`好像不是 rerank -> 对比学习`
> 这个问题本身就是错的，14能回答

15. “接近监督方法”是什么意思？我现在的模型（全监督 Fine-tuning + 检索 + 对比学习）算不算就是一个全监督方法？
> TODO: 是的吧

7. 原始 BCE embedding 在系统中承担什么作用？
> 被使用对比学习这种方式微调。

8. 系统是否分别编码了 term name 和带描述、同义词、父节点的 context？
> TODO: Context text是错的，里边只有name和code，其他的信息都没有。后续还需要更新


9. name embedding 和 context embedding 是如何合并的：拼接、分数相加，还是排名融合？
> TODO: 是排名融合，但是我觉得这一点也是需要更新。应该用name去检索，name跟context任意一种都对

10. 检索得到的 Top-50 是每个视图的 Top-50，还是融合之后的 Top-50？
> TODO: 好像是融合之后的

11. reranker 的输入是什么：ICD-10 与检索出的 ICD-11 pair，还是向量？
> 

12. BCE embedding、对比学习 embedding 和 reranker 是否被拼接或联合训练？
> TODO:没有拼接，就是各自训练各自的 但是我不清楚我有没有用原始的它的embedding去得到的结果

13. 最终测试使用的是原始 BCE embedding，还是微调后的 embedding？
> 用的微调之后的embedding

**C. 训练过程**


16. 硬负例（Hard Negatives）是怎么产生的？
> 在index这个脚本中，相当于检索出来了相似度最高的，但不是正确的，是一种困难样本的产生方法

17. 硬负例是否是“原始 BCE 检索得分很高，但不属于标注正例”的 ICD-11？
> shide 

18. 每个查询究竟使用多少正例和负例：1:4、1:5，还是还包含 batch 内负例？`batch内的负例和其他负例的区别是啥呢`
> 1:5 并且包含 in-batch 负例

19. 模型使用什么损失：BCE、triplet loss、contrastive loss，还是交叉熵形式的 InfoNCE？`这个损失函数是作用在哪里呢？用于fine-tuning BCE嘛？`
> TODO:

20. 微调的是整个 BCE encoder，还是只训练额外的 projection/head？`我甚至看不懂这里说的projection/head 是什么意思`
> TODO:

21. 对比学习如何更新模型，又如何用于之后的向量索引？
> 就是用对比学习更新之后的bce embedding已经不算是bc embedding了，然后用它去做检索

**D. 评测与泄露**
22. 我的全量测试集中，80%是训练集，10%是验证集。这是否意味着测试集里 90% 的数据都是已经训练过的？这算不算数据泄露？
> TODO: 没有泄露，可以看看最新的这个对话

23. 测试集正确的 ICD-11 目标是否在训练集中作为其他 ICD-10 的目标出现过？
> TODO: 没有泄露，可以看看最新的这个对话

24. 目标 ICD-11 重叠是否构成数据泄露？`什么意思`


25. 当前结果能证明的是未见 ICD-10 查询泛化，还是未见 ICD-11 目标泛化？`什么意思`
> 应该是相当于未见ICD 10的查询泛化，因为ICD 10是query，ICD 11是target



**E. 比较与研究定位**
27. BERT-PATH 的候选空间和当前模型是否相同，结果能否直接比较？
> 不能够直接比较，因为测试集的候选集是不一样的

28. 当前“全量 ICD-11 检索”是否比 BERT-PATH 的穷举 pair 评测更困难？`如果我需要将这些结果拿来做对比的话，怎么比较呢？`

29. reranker 能不能替换成 BeLink？
> TODO: 我感觉是可以的，但是如果说rerank替换成belink的话，那创新点在哪里呢

30. BeLink 是一个完整模型、候选生成器，还是可替换的 reranker？
> 我感觉都可以作为吧

31. 如果使用 BeLink，需要建立哪些相同条件下的 baseline？

32. 当前方法的创新点到底是什么？

33. “不依赖目标术语对的训练标签”具体是什么意思？ `33 34 35 36 这几个问题应该`

34. 当前模型是否依赖目标任务 ICD-10→ICD-11 的 gold mapping？

35. “跨五组术语库”是五组任务分别训练测试，还是在 AB 上训练、CD 上零样本测试？
> TODO: 假如说我只是用ICD 10, 然后对他进行一个这种微调的话，那么对于其他数据集呢，还得分别微调，那有没有什么方法是能够跨5组术语库，然后同时对它们进行微调的呢

36. 当前方法属于全监督、弱监督、自监督，还是混合方法？


37. 接下来应该继续做全监督，还是改成弱监督/自监督？

38. 希望“全量候选结果逼平别人较小测试集结果”，是否是合理、可验证的研究目标？

**F. 你的学习和工作方法**
为什么看过一次 Codex 的整体讲解，依旧没有真正理解？

是否应该再次让 Codex 从头解释整个代码？

是否应该自己逐脚本阅读，用自己的语言复述，再让 Codex纠错？

阅读代码应该按文件顺序，还是按数据流顺序？


## 一、ICD10-ICD11的数据
### 1. 原始数据
map数据集
- `original_map_data(只读)/2ICD10-ICD11` 

ICD-11数据集
- `original_term_data（只读）/1_ICD11_crowl`

ICD-10数据集
- `original_term_data（只读）/ICD10-2023`
- `original_term_data（只读）/ICD10CM_202404`
- `original_term_data（只读）/ICD10CM_FY2017_Full_XML`

**注意：**
1. map数据集和 term数据集很可能对应不上。
2. ICD10有是三个版本，需要确认用的是哪个版本。TODO:
3. 需要确认 term中的字段有哪些？map中的字段有哪些？TODO:



### 2. 如何对原始数据进行prepared
codex 生成的处理文档在 `medical_alignment_pipeline/`中，`processed_alignment_data/` 指的是处理好的数据。

**注意：**
1. term 与 map 我希望的字段有没有完全处理。TODO:


### 3. prepared data 的描述。
`entity_linking/data/source`:数据链接，从 `processed_alignment_data/` copy过来的。

**ICD-10字段** : `entity_linking/data/source/who_icd10.jsonl`
term_uid, terminology, edition, version, id, id_type, code, uri, name, description, definitions, synonyms, index_terms, parent_ids, child_ids, path_ids, path_names, class_kind, active, language, source, is_map_derived, raw_fields

其中 raw_fields 内部还包含：

classKind, depth, chapter

**ICD-11** : `entity_linking/data/source/icd11_mms.jsonl`
term_uid, terminology, edition, version, id, id_type, code, uri, name, description, definitions, synonyms, index_terms, parent_ids, child_ids, path_ids, path_names, class_kind, active, language, source, is_map_derived, raw_fields

其中 raw_fields 内部包含：

@context, @id, parent, browserUrl, code, source, classKind, title, indexTerm

而 title 内部包含：

@language, @value

indexTerm 为数组，其元素内部包含：

label

label 内部包含：

@language, @value


**map数据**：`entity_linking/data/source/icd10_icd11.jsonl`

``map_uid``, ``mapping_group``, ``mapping_set``, ``source_term_uid``, ``target_term_uid``, ``source_id``, ``source_name``, ``target_id``, ``target_name``, ``relation``, ``direction``, ``map_group``, ``map_priority``, ``map_rule``, ``map_advice``, ``correlation``, ``mapping_category``, ``active``, ``source_version``, ``target_version``, ``provenance``, ``raw_fields``

raw_fields 内部字段：
``source_file``, ``line_number``, ``columns``

如果需要展开 columns 对应的数据结构（该映射文件实际列定义），则可以表示为：
``source_class_kind``, ``source_depth``, ``source_code``, ``source_chapter``, ``source_title``, ``target_class_kind``, ``target_depth``, ``target_entity_uri``, ``target_release_uri``, ``target_code``, ``target_chapter``, ``target_title``


# 预实验后续问题

1. Belink的关键创新点在于？是否适合我的术语对齐的数据？感觉很适合rerank，如果我用了rerank，那么我的创新点在哪里？相当于是 BCE embedding 粗召回，然后 belink 重召回。
2. Chrome的context_text的信息是错误的，只加载了title 和code，没有同义词，Description 等信息。
3. 不清楚是否有对BCE embedding 没有tuning之前得到其结果
4. 如果 BCE embedding 换一个粗检索的模型呢？会不会有更好的结果。
5. 之前用的RRF融合，但是这样很容易使得两个视图的得到一致结果但错误的 target的权重提高。
6. 我在mac环境下对模型进行了注释，在服务器上run了一次实验。但是两个环境中没有对项目进行git管理。 done



